In [1]:
import polars as pl
import matplotlib
import numpy as np
import math
import pyarrow as pa
import IProgress
import hashlib
import io
from PIL import Image
from pathlib import Path
import plotly.express as px
import nbformat

In [2]:
print(pl.__version__)
print(pa.__version__)

1.40.0
23.0.1


In [3]:
# Import dataset
splits = {'train': 'data/train-00000-of-00001.parquet', 'validation': 'data/validation-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
df_train = pl.read_parquet('hf://datasets/mmenendezg/pneumonia_x_ray/' + splits['train'])
df_test = pl.read_parquet('hf://datasets/mmenendezg/pneumonia_x_ray/' + splits['test'])
df_val = pl.read_parquet('hf://datasets/mmenendezg/pneumonia_x_ray/' + splits['validation'])

The original splits contained overlapping images (based on hashed bytes). By concatenating the three splits and removing duplicate images, I can resplit the data without leakage.

In [4]:
df = pl.concat([df_train, df_test, df_val])

#### First look

In [5]:
df.head

<bound method DataFrame.head of shape: (5_856, 2)
┌─────────────────────────────────┬───────┐
│ image                           ┆ label │
│ ---                             ┆ ---   │
│ struct[2]                       ┆ i64   │
╞═════════════════════════════════╪═══════╡
│ {b"\xff\xd8\xff\xe0\x00\x10JFI… ┆ 0     │
│ {b"\xff\xd8\xff\xe0\x00\x10JFI… ┆ 0     │
│ {b"\xff\xd8\xff\xe0\x00\x10JFI… ┆ 0     │
│ {b"\xff\xd8\xff\xe0\x00\x10JFI… ┆ 0     │
│ {b"\xff\xd8\xff\xe0\x00\x10JFI… ┆ 0     │
│ …                               ┆ …     │
│ {b"\xff\xd8\xff\xe0\x00\x10JFI… ┆ 1     │
│ {b"\xff\xd8\xff\xe0\x00\x10JFI… ┆ 1     │
│ {b"\xff\xd8\xff\xe0\x00\x10JFI… ┆ 1     │
│ {b"\xff\xd8\xff\xe0\x00\x10JFI… ┆ 1     │
│ {b"\xff\xd8\xff\xe0\x00\x10JFI… ┆ 1     │
└─────────────────────────────────┴───────┘>

In [6]:
df.schema

Schema([('image', Struct({'bytes': Binary, 'path': String})),
        ('label', Int64)])

In [7]:
df.group_by("label").len()

label,len
i64,u32
1,4273
0,1583


### Check for duplicates

In [8]:
total_rows = df.height
unique_paths = df.select(pl.col("image").struct.field("path").n_unique()).item()
unique_hashes = df.select(pl.col("image").struct.field("bytes").hash().n_unique()).item()

print("Total rows:", total_rows)
print("Unique paths:", unique_paths)
print("Unique hashes:", unique_hashes)

Total rows: 5856
Unique paths: 4273
Unique hashes: 5383


It seems like a decent number of images use the same path, but only a few of the images are actually identical.

In [9]:
# Add image_hash and image_path columns
df = df.with_columns(pl.col("image").struct.field("bytes").hash().alias("image_hash"))
df = df.with_columns(pl.col("image").struct.field("path").alias("image_path"))
df.head(5)

image,label,image_hash,image_path
struct[2],i64,u64,str
"{b""\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x08\x06\x06\x07\x06\x05\x08\x07\x07\x07\x09\x09\x08\x0a\x0c\x14\x0d\x0c\x0b\x0b\x0c\x19\x12\x13\x0f\x14\x1d\x1a\x1f\x1e\x1d\x1a\x1c\x1c\x20""…,""train-0.jpeg""}",0,15624522136123532950,"""train-0.jpeg"""
"{b""\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x08\x06\x06\x07\x06\x05\x08\x07\x07\x07\x09\x09\x08\x0a\x0c\x14\x0d\x0c\x0b\x0b\x0c\x19\x12\x13\x0f\x14\x1d\x1a\x1f\x1e\x1d\x1a\x1c\x1c\x20""…,""train-1.jpeg""}",0,18221970835360844062,"""train-1.jpeg"""
"{b""\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x08\x06\x06\x07\x06\x05\x08\x07\x07\x07\x09\x09\x08\x0a\x0c\x14\x0d\x0c\x0b\x0b\x0c\x19\x12\x13\x0f\x14\x1d\x1a\x1f\x1e\x1d\x1a\x1c\x1c\x20""…,""train-10.jpeg""}",0,9990647761240806942,"""train-10.jpeg"""
"{b""\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x08\x06\x06\x07\x06\x05\x08\x07\x07\x07\x09\x09\x08\x0a\x0c\x14\x0d\x0c\x0b\x0b\x0c\x19\x12\x13\x0f\x14\x1d\x1a\x1f\x1e\x1d\x1a\x1c\x1c\x20""…,""train-100.jpeg""}",0,10330418483954177345,"""train-100.jpeg"""
"{b""\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x08\x06\x06\x07\x06\x05\x08\x07\x07\x07\x09\x09\x08\x0a\x0c\x14\x0d\x0c\x0b\x0b\x0c\x19\x12\x13\x0f\x14\x1d\x1a\x1f\x1e\x1d\x1a\x1c\x1c\x20""…,""train-1000.jpeg""}",0,13266960183753597657,"""train-1000.jpeg"""


In [10]:
exact_dupes = df.filter(
    pl.struct(["image_hash", "label", "image_path"]).is_duplicated()
)
len(exact_dupes)

0

In [11]:
hash_dupes = df.filter(pl.struct(["image_hash"]).is_duplicated())
len(hash_dupes)

937

In [12]:
# Find out if any duplicate hashes have different labels
dup_hashes = (
    df.group_by("image_hash")
    .agg([
        pl.col("image_path").n_unique().alias("unique_paths"),
        pl.col("label").n_unique().alias("unique_labels")
    ])
    .filter(pl.col("unique_paths") > 1)
)

print("Duplicate hashes:", len(dup_hashes))


Duplicate hashes: 464


In [13]:
dup_hashes.sort(pl.col("unique_labels"), descending=True)

image_hash,unique_paths,unique_labels
u64,u32,u32
5192626226014858891,2,1
15389504758763153875,2,1
17178489390230692853,2,1
18068287191877136842,2,1
3216275527068186822,2,1
…,…,…
6367729287435314602,2,1
8975624326262888069,2,1
12403435077334984823,3,1


It appears that duplicate image hashes all have identical labels, just unique paths. Because there is not overlap, it is safe to drop the duplicates.

In [14]:
# Drop the duplicates
df = df.unique(subset='image_hash')

In [15]:
print("Rows after removing duplicates: ", df.height)
print(df.group_by("label").len())

Rows after removing duplicates:  5383
shape: (2, 2)
┌───────┬──────┐
│ label ┆ len  │
│ ---   ┆ ---  │
│ i64   ┆ u32  │
╞═══════╪══════╡
│ 1     ┆ 3916 │
│ 0     ┆ 1467 │
└───────┴──────┘


### Saving images, renaming files to hashes

In [16]:
# New folder for images
out_dir = Path("data/clean_xrays")
out_dir.mkdir(parents=True, exist_ok=True)

In [17]:
# Shuffle the data
df = df.sample(fraction=1, seed=12)

In [18]:
df.schema

Schema([('image', Struct({'bytes': Binary, 'path': String})),
        ('label', Int64),
        ('image_hash', UInt64),
        ('image_path', String)])

In [19]:
def save_image(image_struct, image_hash):
    """
    Save an image from raw bytes to disk using a hash-based filename.
    Returns file path to the image
    """
    image_bytes = image_struct['bytes']
    out_path = (out_dir / f"{image_hash}.png")

    if not out_path.exists():
        img = Image.open(io.BytesIO(image_bytes))
        img.save(out_path)
    return out_path.as_posix()


In [20]:
df = df.with_columns(
    pl.struct(["image", "image_hash"])
    .map_elements(
        lambda row: save_image(row["image"], row["image_hash"]),
        return_dtype=pl.String
    )
    .alias("file_path")
)

In [21]:
df = df.select(['file_path', 'image_hash', 'label'])

In [22]:
df.head

<bound method DataFrame.head of shape: (5_383, 3)
┌─────────────────────────────────┬──────────────────────┬───────┐
│ file_path                       ┆ image_hash           ┆ label │
│ ---                             ┆ ---                  ┆ ---   │
│ str                             ┆ u64                  ┆ i64   │
╞═════════════════════════════════╪══════════════════════╪═══════╡
│ data/clean_xrays/2227043032124… ┆ 2227043032124966785  ┆ 0     │
│ data/clean_xrays/6083004423535… ┆ 6083004423535372348  ┆ 1     │
│ data/clean_xrays/1783811969902… ┆ 17838119699026671587 ┆ 0     │
│ data/clean_xrays/1126480666513… ┆ 11264806665138737235 ┆ 0     │
│ data/clean_xrays/1138531225824… ┆ 11385312258241916384 ┆ 1     │
│ …                               ┆ …                    ┆ …     │
│ data/clean_xrays/8335306669402… ┆ 8335306669402464873  ┆ 1     │
│ data/clean_xrays/7510579336723… ┆ 751057933672326025   ┆ 1     │
│ data/clean_xrays/7766798036395… ┆ 7766798036395567545  ┆ 1     │
│ data/clean

### Train/Test/val split

In [23]:
n = df.height

In [24]:
train_end = int(0.7*n)
val_end = int(0.9*n)

In [25]:
train_df = df[:train_end]
val_df = df[train_end:val_end]
test_df = df[val_end:]

In [26]:
train_counts = (
    train_df.group_by("label")
    .len()
    .with_columns(
        (pl.col("len") / train_df.height).alias("proportion")
    )
)
fig = px.pie(
    train_counts,
    names="label",
    values="proportion",
    title="Train Split Class Distribution"
)
fig.show()

In [27]:
test_counts = (
    test_df.group_by("label")
    .len()
    .with_columns(
        (pl.col("len") / test_df.height).alias("proportion")
    )
)
fig = px.pie(
    test_counts,
    names="label",
    values="proportion",
    title="Test Split Class Distribution"
)
fig.show()

In [28]:
val_counts = (
    val_df.group_by("label")
    .len()
    .with_columns(
        (pl.col("len") / val_df.height).alias("proportion")
    )
)
fig = px.pie(
    val_counts,
    names="label",
    values="proportion",
    title="Validation Split Class Distribution"
)
fig.show()

In [29]:
train_hashes = set(train_df["image_hash"])
val_hashes = set(val_df["image_hash"])
test_hashes = set(test_df["image_hash"])

In [30]:
print("train-val overlap:", len(train_hashes & val_hashes))
print("train-test overlap:", len(train_hashes & test_hashes))
print("val-test overlap:", len(val_hashes & test_hashes))

train-val overlap: 0
train-test overlap: 0
val-test overlap: 0


In [31]:
# label train/val images, concat into one df for fastai
train_df = train_df.with_columns(val = False)
val_df = val_df.with_columns(val = True)
df = pl.concat(
    [
        train_df,
        val_df,
    ],
    how="vertical"
)
df.head

<bound method DataFrame.head of shape: (4_844, 4)
┌─────────────────────────────────┬──────────────────────┬───────┬───────┐
│ file_path                       ┆ image_hash           ┆ label ┆ val   │
│ ---                             ┆ ---                  ┆ ---   ┆ ---   │
│ str                             ┆ u64                  ┆ i64   ┆ bool  │
╞═════════════════════════════════╪══════════════════════╪═══════╪═══════╡
│ data/clean_xrays/2227043032124… ┆ 2227043032124966785  ┆ 0     ┆ false │
│ data/clean_xrays/6083004423535… ┆ 6083004423535372348  ┆ 1     ┆ false │
│ data/clean_xrays/1783811969902… ┆ 17838119699026671587 ┆ 0     ┆ false │
│ data/clean_xrays/1126480666513… ┆ 11264806665138737235 ┆ 0     ┆ false │
│ data/clean_xrays/1138531225824… ┆ 11385312258241916384 ┆ 1     ┆ false │
│ …                               ┆ …                    ┆ …     ┆ …     │
│ data/clean_xrays/5198290170582… ┆ 5198290170582868336  ┆ 1     ┆ true  │
│ data/clean_xrays/8659261141483… ┆ 86592611414836

In [32]:
from PIL import Image
img = Image.open(df['file_path'][0])
print(img.size)
print(img.mode)

(500, 500)
RGB


In [33]:
df_pandas = df.to_pandas()

In [34]:
df_pandas.head()

,file_path,image_hash,label,val
0,data/clean_xrays/2227043032124966785.png,2227043032124966785,0,False
1,data/clean_xrays/6083004423535372348.png,6083004423535372348,1,False
2,data/clean_xrays/17838119699026671587.png,17838119699026671587,0,False
3,data/clean_xrays/11264806665138737235.png,11264806665138737235,0,False
4,data/clean_xrays/11385312258241916384.png,11385312258241916384,1,False


### Fastai/modeling

In [35]:
from fastai.vision.all import *
import torch
import fastai

In [36]:
dls = ImageDataLoaders.from_df(
    df_pandas,
    path='.',
    fn_col='file_path',
    label_col='label',
    valid_col='val',
    item_tfms=Resize(224),
    batch_tfms=[*aug_transforms(
        do_flip=False,
        flip_vert=False,
        max_rotate=20,
        max_zoom=1.2,
        max_warp=0.2,
        max_lighting=0.2,
        ),
        Normalize.from_stats(*imagenet_stats)
    ]
)

Transforms loosely based on the parameters provided in table 3 of "Pneumonia Detection from Chest X-Ray Images Using Deep Learning and Transfer Learning for Imbalanced Datasets", found here: https://pmc.ncbi.nlm.nih.gov/articles/PMC12344030/#Sec22

In [37]:
model_path = Path('pneumonia_learner.pkl')

if model_path.exists():
    learn = load_learner(model_path)
else:
    learn = vision_learner(
        dls,
        resnet34,
        metrics=[accuracy, Precision(), Recall(), F1Score()]
    )
    learn.fine_tune(5)
    learn.export(model_path)

c:\Users\chris\PycharmProjects\Chest_xrays\.venv\Lib\site-packages\fastai\learner.py:455: UserWarning: load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.
If you only need to load model weights and optimizer state, use the safe `Learner.load` instead.
  warn("load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.\nIf you only need to load model weights and optimizer state, use the safe `Learner.load` instead.")


In [60]:
print(type(learn.loss_func))
print(learn.dls)
print(learn.dls.valid)

<class 'fastai.losses.CrossEntropyLossFlat'>


In [50]:
x,y = dls.one_batch()

In [51]:
y

TensorCategory([1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1,
                1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1,
                0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1,
                0], device='cuda:0')

In [52]:
preds,_ = learn.get_preds(dl=[(x,y)])
preds[0].numpy().round(5)

array([9.6000e-04, 9.9904e-01], dtype=float32)